In [1]:
import torch

# Check for NVIDIA GPU (CUDA)
print("CUDA Available:", torch.cuda.is_available())

# Check for Apple Silicon Mac GPU (MPS)
print("MPS Available:", torch.backends.mps.is_available())

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

CUDA Available: True
MPS Available: False
Using device: cuda


In [2]:
import pandas as pd

family_df = pd.read_csv('protein_families.csv')

i = 0
name, accession, match_len, seed_len, type_, clan = family_df.iloc[i]

In [3]:
from Bio import SeqIO
from modules.sequences import split_train_val_test

# read the fasta files
recs = [(r.id, str(r.seq)) for r in SeqIO.parse(f"sequences/{accession}.fasta", "fasta")]

# separate to train, validation, and test set
train_seqs, val_seqs, test_seqs = split_train_val_test(recs)

In [4]:
from modules.qmodel import initialize_random_hqmm

n = match_len * 2 + 1
W = 1
s = 21
p, K = initialize_random_hqmm(n, s, W, 'cpu')

In [5]:
from modules.model import (
    init_transition_mat,
    init_emission_mat
)

# init transition and emmission matrices
trans_matdf = init_transition_mat(match_len)
emit_matdf = init_emission_mat(match_len, seed=2)

# encode the symbols into index for the hmmlearn
symbols = list(emit_matdf.index)            # ^ A C D ... Y $  (taken from the CSV row order)
sym_idx = {s: i for i, s in enumerate(symbols)}
BEGIN = sym_idx["^"]                         # dummy begin symbol, emitted only by M0
END = sym_idx["$"]    

In [6]:
from modules.qmodel import convert_to_torch

K_torch = convert_to_torch(K)

/home/kyle/github/Profile-Hidden-Quantum-Markov-Model/modules/qmodel.py:220: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  key: torch.tensor(


In [8]:
import torch
import numpy as np
from pathlib import Path

from modules.qmodel import (
    initialize_random_hqmm,
    convert_to_torch
)
from modules.qtrain import train_hqmm
from modules.qevaluation import description_accuracy_hqmm
from modules.model import (
    init_transition_mat,
    init_emission_mat
)

# ==========================================
# 1. MODEL CONFIGURATION
# ==========================================

n = match_len * 2 + 1
s = 21  # 20 amino acids + $

# Initialize transition and emission matrices
trans_matdf = init_transition_mat(match_len)
emit_matdf = init_emission_mat(match_len, seed=2)

symbols = list(emit_matdf.index)
sym_idx = {symbol: i for i, symbol in enumerate(symbols)}

BEGIN = sym_idx["^"]
END = sym_idx["$"]

# Random initialization assumes modeled symbol IDs are 1...s
assert BEGIN == 0
assert sorted(
    idx for symbol, idx in sym_idx.items()
    if symbol != "^"
) == list(range(1, s + 1))

# ==========================================
# 2. TRAINING CONFIGURATION
# ==========================================

# Vary the number of Kraus operators
settings = {
    f"w={w}": w
    for w in sorted(set([2, int(n-70), int(n-50), int(n-30)]))
}

# Fixed optimization parameters
tau = 0.1
alpha = 0.8
max_iter = 15
tol = 1e-3
batch_size = 16

# Results containers
results = {}
histories = {}
trained_models = {}

# Directory for saved models
model_dir = Path("saved_models/phqmm/w-random")
model_dir.mkdir(parents=True, exist_ok=True)

# ==========================================
# 3. FIXED INITIAL DENSITY MATRIX
# ==========================================

# Keep p identical across all W configurations
torch.manual_seed(2)

G = (
    torch.randn((n, n), dtype=torch.float32)
    + 1j * torch.randn((n, n), dtype=torch.float32)
).to(torch.complex64)

p = G @ G.conj().T
p = p / torch.trace(p).real

# ==========================================
# 4. TRAINING LOOP
# ==========================================

for label, W in settings.items():

    print(
        f"--- {label}: n={n}, W={W}, tau={tau}",
        flush=True
    )

    # Initialize NEW Kraus operators for each W
    torch.manual_seed(100 + W)

    _, K = initialize_random_hqmm(
        n=n,
        s=s,
        w=W,
        device="cpu"
    )

    K_torch = convert_to_torch(K)

    K_initial = {
        key: value.detach().clone()
        for key, value in K_torch.items()
    }

    # --------------------------------------
    # Evaluate untrained model
    # --------------------------------------

    K_ref = next(iter(K_initial.values()))

    p_eval = p.to(
        device=K_ref.device,
        dtype=K_ref.dtype
    )

    with torch.no_grad():
        da0 = description_accuracy_hqmm(
            p_eval,
            K_initial,
            W,
            sym_idx,
            test_seqs
        )

    results[f"{label} (untrained)"] = (
        float(da0.mean()),
        float(da0.std())
    )

    # --------------------------------------
    # Train HQMM
    # --------------------------------------

    K_trained, hist = train_hqmm(
        p=p,
        K=K_initial,
        W=W,
        sym_idx=sym_idx,
        train_seqs=train_seqs,
        val_seqs=val_seqs,
        test_seqs=test_seqs,
        max_iter=max_iter,
        tol=tol,
        tau=tau,
        alpha=alpha,
        batch_size=batch_size,
        verbose=True,
        device=device
    )

    histories[label] = hist
    trained_models[label] = K_trained

    # --------------------------------------
    # Save checkpoint and history
    # --------------------------------------

    model_path = model_dir / (
        f"{name}_{accession}_hqmm_"
        f"w{W}_tau_{tau:g}_alpha_{alpha:g}.pt"
    )

    history_path = model_dir / (
        f"{name}_{accession}_history_"
        f"w{W}_tau_{tau:g}_alpha_{alpha:g}.csv"
    )

    checkpoint = {
        "K": {
            key: value.detach().cpu().clone()
            for key, value in K_trained.items()
        },
        "p": p.detach().cpu().clone(),
        "n": n,
        "s": s,
        "W": W,
        "tau": tau,
        "alpha": alpha,
        "batch_size": batch_size,
        "max_iter": max_iter,
        "tol": tol,
        "epochs_completed": len(hist) - 1,
    }

    torch.save(checkpoint, model_path)
    hist.to_csv(history_path, index=False)

    print(f"Model saved: {model_path}", flush=True)
    print(f"History saved: {history_path}", flush=True)

    # --------------------------------------
    # Evaluate trained model
    # --------------------------------------

    K_ref = next(iter(K_trained.values()))

    p_eval = p.to(
        device=K_ref.device,
        dtype=K_ref.dtype
    )

    K_eval = {
        key: value.to(
            device=K_ref.device,
            dtype=K_ref.dtype
        )
        for key, value in K_trained.items()
    }

    with torch.no_grad():
        da = description_accuracy_hqmm(
            p_eval,
            K_eval,
            W,
            sym_idx,
            test_seqs
        )

    results[label] = (
        float(da.mean()),
        float(da.std())
    )

    print(
        f"Test DA: {results[label][0]:.4f} "
        f"± {results[label][1]:.4f}",
        flush=True
    )

# ==========================================
# 5. FINAL SUMMARY
# ==========================================

print("\nFinal Test DA Results")

for label, (mean, std) in results.items():
    print(f"{label}: {mean:.4f} ± {std:.4f}")


--- w=2: n=79, W=2, tau=0.1


/home/kyle/github/Profile-Hidden-Quantum-Markov-Model/modules/qmodel.py:220: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  key: torch.tensor(


Training on cuda (dtype=torch.complex64)
iter   0  train     0.0s  log-lik -14665.46  train_da_mean 0.0002  val_da_mean 0.0003
  iter   1  batch   1/5  loss 235.6403  tau 0.100000
  iter   1  batch   2/5  loss 196.8109  tau 0.100000
  iter   1  batch   3/5  loss 180.3809  tau 0.100000
  iter   1  batch   4/5  loss 205.5353  tau 0.100000
  iter   1  batch   5/5  loss 193.7365  tau 0.100000
iter   1  train     5.8s  log-lik -13703.31  train_da_mean 0.0665  val_da_mean 0.0614
  iter   2  batch   1/5  loss 179.2830  tau 0.080000
  iter   2  batch   2/5  loss 195.7174  tau 0.080000
  iter   2  batch   3/5  loss 193.0772  tau 0.080000
  iter   2  batch   4/5  loss 201.3130  tau 0.080000
  iter   2  batch   5/5  loss 211.1616  tau 0.080000
iter   2  train    11.2s  log-lik -13228.21  train_da_mean 0.1001  val_da_mean 0.0896
  iter   3  batch   1/5  loss 193.2314  tau 0.064000
  iter   3  batch   2/5  loss 173.9518  tau 0.064000
  iter   3  batch   3/5  loss 188.7823  tau 0.064000
  iter   3  

/home/kyle/github/Profile-Hidden-Quantum-Markov-Model/modules/qmodel.py:220: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  key: torch.tensor(


Training on cuda (dtype=torch.complex64)
iter   0  train     0.0s  log-lik -14663.01  train_da_mean 0.0002  val_da_mean 0.0003
  iter   1  batch   1/5  loss 201.6757  tau 0.100000
  iter   1  batch   2/5  loss 228.0636  tau 0.100000
  iter   1  batch   3/5  loss 190.8035  tau 0.100000
  iter   1  batch   4/5  loss 203.3594  tau 0.100000
  iter   1  batch   5/5  loss 186.5060  tau 0.100000
iter   1  train    21.1s  log-lik -13901.62  train_da_mean 0.0521  val_da_mean 0.0494
  iter   2  batch   1/5  loss 240.0305  tau 0.080000
  iter   2  batch   2/5  loss 195.2715  tau 0.080000
  iter   2  batch   3/5  loss 189.9756  tau 0.080000
  iter   2  batch   4/5  loss 170.0185  tau 0.080000
  iter   2  batch   5/5  loss 186.0493  tau 0.080000
iter   2  train    39.1s  log-lik -13746.51  train_da_mean 0.0631  val_da_mean 0.0596
  iter   3  batch   1/5  loss 234.1557  tau 0.064000
  iter   3  batch   2/5  loss 184.3363  tau 0.064000
  iter   3  batch   3/5  loss 188.6396  tau 0.064000
  iter   3  

/home/kyle/github/Profile-Hidden-Quantum-Markov-Model/modules/qmodel.py:220: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  key: torch.tensor(


Training on cuda (dtype=torch.complex64)
iter   0  train     0.0s  log-lik -14665.06  train_da_mean 0.0000  val_da_mean 0.0000
  iter   1  batch   1/5  loss 187.0473  tau 0.100000
  iter   1  batch   2/5  loss 215.5355  tau 0.100000
  iter   1  batch   3/5  loss 195.2972  tau 0.100000
  iter   1  batch   4/5  loss 226.3308  tau 0.100000
  iter   1  batch   5/5  loss 187.5534  tau 0.100000
iter   1  train    47.6s  log-lik -13926.56  train_da_mean 0.0504  val_da_mean 0.0477
  iter   2  batch   1/5  loss 176.3365  tau 0.080000
  iter   2  batch   2/5  loss 193.8986  tau 0.080000
  iter   2  batch   3/5  loss 212.4726  tau 0.080000
  iter   2  batch   4/5  loss 214.5042  tau 0.080000
  iter   2  batch   5/5  loss 185.9955  tau 0.080000
iter   2  train    95.7s  log-lik -13787.21  train_da_mean 0.0600  val_da_mean 0.0571
  iter   3  batch   1/5  loss 175.5127  tau 0.064000
  iter   3  batch   2/5  loss 200.2189  tau 0.064000
  iter   3  batch   3/5  loss 176.4706  tau 0.064000
  iter   3  

/home/kyle/github/Profile-Hidden-Quantum-Markov-Model/modules/qmodel.py:220: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  key: torch.tensor(


Training on cuda (dtype=torch.complex64)
iter   0  train     0.0s  log-lik -14665.35  train_da_mean 0.0000  val_da_mean 0.0000
  iter   1  batch   1/5  loss 191.6149  tau 0.100000
  iter   1  batch   2/5  loss 226.2243  tau 0.100000
  iter   1  batch   3/5  loss 205.3166  tau 0.100000
  iter   1  batch   4/5  loss 207.3056  tau 0.100000
  iter   1  batch   5/5  loss 171.8978  tau 0.100000
iter   1  train    83.8s  log-lik -13934.39  train_da_mean 0.0497  val_da_mean 0.0477
  iter   2  batch   1/5  loss 210.9252  tau 0.080000
  iter   2  batch   2/5  loss 189.0887  tau 0.080000
  iter   2  batch   3/5  loss 211.1696  tau 0.080000
  iter   2  batch   4/5  loss 195.2024  tau 0.080000
  iter   2  batch   5/5  loss 162.2817  tau 0.080000
iter   2  train   167.4s  log-lik -13792.01  train_da_mean 0.0598  val_da_mean 0.0572
  iter   3  batch   1/5  loss 224.2699  tau 0.064000
  iter   3  batch   2/5  loss 191.5250  tau 0.064000
  iter   3  batch   3/5  loss 187.0822  tau 0.064000
  iter   3  

In [9]:
import pandas as pd
import matplotlib.pyplot as plt

from modules.qplots import (
    plot_da_learning_curve,
    plot_da_bars,
)

from pathlib import Path

try:
    plt.style.use("seaborn-v0_8-darkgrid")
except OSError:
    pass

# Create directory for saved models
results_dir = Path("results/phqmm/w-random")

# Save training histories
for label, h in histories.items():
    h.to_csv(
        f"q_history_{label.split()[0]}.csv",
        index=False
    )

# Save combined training histories
combined_histories = pd.concat(
    [
        h.assign(step_size=label)
        for label, h in histories.items()
    ],
    ignore_index=True,
)

combined_histories.to_csv(
    results_dir / f"{name}_q_training_histories.csv",
    index=False,
)

# Save final description accuracy results
pd.DataFrame(
    results,
    index=["test_da_mean", "test_da_std"]
).T.to_csv(results_dir / f"{name}_q_da_results.csv")

# Optional reference DA values
references = {}

# Test DA vs GPU training time
plot_da_learning_curve(
    histories,
    references,
    x="train_time",
    out=results_dir / f"{name}_q_da_vs_time.png"
)

# Test DA vs epoch
plot_da_learning_curve(
    histories,
    references,
    x="iter",
    out=results_dir / f"{name}_q_da_vs_epoch.png"
)

# Final DA comparison by step size
plot_da_bars(
    results,
    references,
    out=results_dir / f"{name}_q_da_results.png"
)